# Etude pour afficher un résumé des logs.
Ne fonctionne qu'avec des fichiers bien structurés. Ne fonctionne qu'avec des dates en français (!)

In [ ]:
# Simplifie le fichier de type ls -ltR sur les logs de Glims
# Idée : produire un fichier html  de synthèse.

from datetime import datetime, timedelta
import re
from pprint import  pprint
from pathlib import Path
from IPython.display import display, HTML

In [ ]:
def parse_ls_lr(contenu):
    resultats = {}
    repertoire_courant = None

    for ligne in contenu.splitlines():
        ligne = ligne.strip()

        # Détection d'un répertoire
        if ligne.startswith("./") and ligne.endswith(":"):
            repertoire_courant = ligne[:-1]  # enlever le ":"
            repertoire_courant = repertoire_courant[2:]
            resultats[repertoire_courant] = None
            continue

        # Détection d'un fichier
        if ligne.startswith("-") and repertoire_courant:
            # Exemple de date : "17 mars 18:40"
            match = re.search(r'(\d{1,2}) (\w+) +(\d{1,2}:\d{2})', ligne)
            if match:
                jour, mois, heure = match.groups()

                # convertir mois FR -> numéro
                mois_fr = {
                    "janv": 1, "févr": 2, "mars": 3, "avr": 4,
                    "mai": 5, "juin": 6, "juil": 7, "août": 8,
                    "sept": 9, "oct": 10, "nov": 11, "déc": 12
                }

                mois_num = mois_fr.get(mois[:4].lower())
                if mois_num is None:
                    continue

                date_obj = datetime(
                    year= datetime.now().year,
                    month=mois_num,
                    day=int(jour),
                    hour=int(heure.split(":")[0]),
                    minute=int(heure.split(":")[1])
                )

                # garder la plus récente
                if (resultats[repertoire_courant] is None or
                        date_obj > resultats[repertoire_courant]):
                    resultats[repertoire_courant] = date_obj

    return resultats


In [ ]:
with open(Path("//sn1314/mips/tempo_bma/svc_logs.txt"), 'r') as f:
# with open("logs/svc_logs.txt", 'r') as f:
        lines = f.read()
        result = parse_ls_lr(lines)
pprint(result)

In [ ]:
len(result)

In [ ]:
def filtrer_anciens(donnees, minutes=15):
    maintenant = datetime.now()
    seuil = timedelta(minutes=minutes)

    resultats = {}

    # for rep, date in donnees.items():
    for rep in sorted(donnees):
        date = donnees[rep]
        if date is None:
            continue

        age = maintenant - date

        if age > seuil:
            resultats[rep] = {
                "date": date,
                "age_minutes": int(age.total_seconds() // 60)
            }

    return resultats

In [ ]:
def dict_to_html(data):
    html = []

    html.append("<table border='1' cellpadding='5' cellspacing='0'>")
    html.append("<tr><th>Répertoire</th><th>Dernière mise à jour</th><th>Âge (minutes)</th></tr>")

    for rep, info in data.items():
        date = info["date"]
        age = info["age_minutes"]

        # couleur si trop ancien
        couleur = "#ffcccc" if age > 53 else "#ccffcc"

        html.append(
            f"<tr style='background-color:{couleur};'>"
            f"<td>{rep}</td>"
            f"<td>{date.strftime('%d/%m/%Y %H:%M')}</td>"
            f"<td>{age}</td>"
            f"</tr>"
        )

    html.append("</table>")

    return "\n".join(html)

In [ ]:
old = filtrer_anciens(result, minutes = 120)

In [ ]:
HTML(dict_to_html(old))